<a href="https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Unit of analysis:** one row = one content item, on one day, for one client
(page × day × client), in `fact_content_daily_performance`.

**Time window:** `month=2026-03` — a mid-panel month, not the sealed final month
(`_sample`/June 2026), per the warehouse warning that the last month is the natural
outcome window for any past→future label and shouldn't be used to develop label logic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Feature:** `days_since_last_update`, `impressions`, `avg_position`, `ctr`,
`content_type` (from a join to `dim_content`), `word_count` — all knowable at the
decision moment (the day an editor would review the page).

**Label/proxy:** a "declining" indicator built the same way as before — a >20% drop in
impressions comparing two trailing windows — but computed properly from
`fact_content_daily_performance`'s own daily rows within March, not borrowed from a
future window.

**Context:** `client_id`, `content_id` — used only for grouping/joins and client-holdout
splits, never as model features (per the data dictionary's ID rule).

**Excluded:** anything from `fact_content_query_90d`. Why: that table's 90-day window
overlaps the most recent months of the snapshot, so its `impressions_90d`/`*_last30`
columns would contain the same period the label is defined over — using them risks
leaking the label's own outcome window into the features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Secret created, ready to query.")

Secret created, ready to query.


In [21]:
con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [22]:
col_names = con.sql("""
    SELECT column_name
    FROM (DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'))
""").df()["column_name"].tolist()

for c in col_names:
    print(c)

report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [23]:
con.sql("""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   n   │
│     varchar     │    varchar     │    date     │ int64 │
├─────────────────┴────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [24]:
con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



In [25]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │ gsc_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │             413966 │            3611061 │
└────────────┴────────────────────┴────────────────────┘



In [26]:
monthly = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_sum_position * 1.0 / gsc_impressions END) AS avg_position,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impressions_second_half,
        SUM(ga4_engaged_sessions) AS engaged_sessions_month,
        SUM(ga4_sessions) AS sessions_month
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) > 0
""").df()

print(monthly.shape)
monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 9)


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position,impressions_first_half,impressions_second_half,engaged_sessions_month,sessions_month
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.714744,18.0,39.0,0.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,4173.0,2350.0,0.0,1.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,6.481453,89.0,60.0,0.0,4.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,2.987198,245.0,208.0,0.0,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,3705.0,1925.0,0.0,3.0


In [27]:
import pandas as pd

# Build the label: declining = second half of March meaningfully lower than first half
monthly["ctr_month"] = (monthly["clicks_month"] / monthly["impressions_month"].replace(0, pd.NA)) * 100
monthly["pct_change"] = (
    (monthly["impressions_second_half"] - monthly["impressions_first_half"])
    / monthly["impressions_first_half"].replace(0, pd.NA)
) * 100
monthly["is_declining_label"] = (monthly["pct_change"] < -20).astype(int)

print(f"Share declining: {monthly['is_declining_label'].mean()*100:.1f}%")

features = monthly[[
    "content_hash_id", "client_hash_id",
    "avg_position", "ctr_month", "sessions_month",
    "engaged_sessions_month", "impressions_first_half",
    "is_declining_label"
]].copy()

features.head()

Share declining: 28.1%


,content_hash_id,client_hash_id,avg_position,ctr_month,sessions_month,engaged_sessions_month,impressions_first_half,is_declining_label
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2.714744,0.000000,0.0,0.0,18.0,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.209549,0.107313,1.0,0.0,4173.0,1
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,6.481453,0.000000,4.0,0.0,89.0,1
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2.987198,0.000000,0.0,0.0,245.0,0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.724039,0.106572,3.0,0.0,3705.0,1


In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

def quick_score(feature_cols, label="honest"):
    X = features[feature_cols].fillna(0)
    y = features["is_declining_label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    print(f"[{label}] accuracy on held-out test: {score:.3f}")
    return score

# HONEST version — only the 5 legitimate features
honest_cols = ["avg_position", "ctr_month", "sessions_month", "engaged_sessions_month", "impressions_first_half"]
quick_score(honest_cols, label="honest")

[honest] accuracy on held-out test: 0.583


0.582814680698578

In [29]:
# THE TRAP — add impressions_second_half, which is literally half the label's own definition
features["impressions_second_half"] = monthly["impressions_second_half"]
leaky_cols = honest_cols + ["impressions_second_half"]
quick_score(leaky_cols, label="LEAKY (on purpose)")

[LEAKY (on purpose)] accuracy on held-out test: 0.799


0.7989702387688129

In [30]:
# DELETE the leak, keep the honest number
features = features.drop(columns=["impressions_second_half"])
print("Leak column removed. Honest features restored:", honest_cols)

Leak column removed. Honest features restored: ['avg_position', 'ctr_month', 'sessions_month', 'engaged_sessions_month', 'impressions_first_half']


**The trap, sprung:** Adding `impressions_second_half` — literally one of the two halves
used to define `is_declining_label` — jumped held-out accuracy from 0.583 to 0.799. This
increase is not a real modeling win; it's the model partially re-deriving its own label
from a feature that overlaps the label's definition. In production, `impressions_second_half`
would not exist yet at decision time (it's the *outcome* period, not something knowable
when an editor reviews a page) — so this "improvement" would be entirely fake and the
model would fail immediately in the real world. The honest 0.583, built only from
features knowable at decision time, is the number that actually matters.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Named limitation:** GSC availability in this March slice is only ~37% of rows
(3,611,061 / 9,841,378), and GA4 availability is only ~4% (413,966 / 9,841,378). This
means most of this month's rows can't be used for GA4-based features at all, and even
GSC-based features only reliably cover about a third of the panel. This is very likely
the "unbalanced panel" issue flagged in the data dictionary — different clients have
different history start dates — rather than a sign that most pages have no real
performance. Any model or finding from this slice describes the ~37% of rows with GSC
coverage, not the full warehouse population, and should not be assumed to generalize to
clients with shorter or missing history in March.

## Self-check

Before you submit, confirm each line honestly:

- [Y] Every section above is filled — markdown thinking AND the code that backs it
- [ Y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Y ] No client names, URLs, or private queries anywhere
- [Y ] My claims use careful words: observed, measured, directional, decision-support
- [ Y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.